In [1]:

import requests
import ollama
from bs4 import BeautifulSoup
from pypdf import PdfReader

MODEL = "gemma4"

SYSTEM = (
    "You summarize text faithfully. Do not add facts that are not in "
    "the source. If information is missing or unclear, say so."
)


def ask(prompt):
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": prompt}
        ],
        options={"temperature": 0.2}
    )
    return response["message"]["content"]


def chunk_text(text, size=6000):
    return [
        text[i:i + size]
        for i in range(0, len(text), size)
    ]


def summarize(text, style="concise bullet points"):
    if not text or not text.strip():
        return "No text available to summarize."

    chunks = chunk_text(text)
    partials = []

    # Summarize each chunk
    for index, chunk in enumerate(chunks, start=1):
        print(f"Summarizing section {index}/{len(chunks)}...")
        partials.append(
            ask(
                f"Summarize this source section faithfully. "
                f"Do not invent facts.\n\n{chunk}"
            )
        )

    # Combine the summaries
    if len(partials) == 1:
        combined = partials[0]
    else:
        combined = "\n\n".join(partials)

    return ask(
        f"Create one {style} summary from the source material below. "
        f"Preserve important facts and avoid repetition. "
        f"Do not add unsupported claims.\n\n{combined}"
    )


def read_pdf(path):
    reader = PdfReader(path)
    return "\n".join(
        page.extract_text() or ""
        for page in reader.pages
    )


def read_url(url):
    response = requests.get(
        url,
        timeout=15,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Remove common non-content elements
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    return soup.get_text(" ", strip=True)


if __name__ == "__main__":
    sample = (
        "Machine learning is a field of AI where computers learn "
        "patterns from data. "
    ) * 20

    print("\n--- TEXT SUMMARY ---")
    print(summarize(sample))

    # Summarize a PDF:
    # print(summarize(read_pdf("file.pdf")))

    # Summarize a webpage:
    # print(summarize(read_url("https://example.com")))



--- TEXT SUMMARY ---
Summarizing section 1/1...
*   Machine learning is a field of AI that enables computers to learn patterns from data.


In [2]:
%pip install rouge-score

Note: you may need to restart the kernel to use updated packages.


In [3]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

print("ROUGE evaluation library installed successfully!")

ROUGE evaluation library installed successfully!


In [4]:

import ollama
from rouge_score import rouge_scorer
import pandas as pd
import time

MODEL = "gemma4"

# Source text and a human-written reference summary
tests = [
    {
        "source": (
            "Machine learning is a branch of artificial intelligence. "
            "It enables computers to learn patterns from data and make "
            "predictions without being explicitly programmed for every task."
        ),
        "reference": (
            "Machine learning is a branch of AI that learns patterns "
            "from data to make predictions."
        )
    },
    {
        "source": (
            "Python is a programming language widely used in data science. "
            "Pandas helps manipulate datasets, NumPy supports numerical "
            "computing, and Matplotlib creates visualizations."
        ),
        "reference": (
            "Python supports data science. Pandas manipulates data, "
            "NumPy performs numerical computing, and Matplotlib creates charts."
        )
    },
    {
        "source": (
            "A train-test split separates data into training and testing sets. "
            "The model learns from training data, while test data estimates "
            "how well it performs on unseen examples."
        ),
        "reference": (
            "Train-test splitting trains a model on one set and evaluates "
            "its performance on unseen test data."
        )
    }
]

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

results = []

for i, test in enumerate(tests, start=1):
    start = time.perf_counter()

    response = ollama.chat(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Summarize faithfully using only the source text. "
                    "Do not add unsupported facts. Keep the summary concise."
                )
            },
            {
                "role": "user",
                "content": "Summarize this text:\n" + test["source"]
            }
        ],
        options={"temperature": 0}
    )

    summary = response["message"]["content"].strip()
    elapsed = time.perf_counter() - start

    scores = scorer.score(test["reference"], summary)

    result = {
        "Test": i,
        "Generated Summary": summary,
        "ROUGE-1": scores["rouge1"].fmeasure,
        "ROUGE-2": scores["rouge2"].fmeasure,
        "ROUGE-L": scores["rougeL"].fmeasure,
        "Time (seconds)": round(elapsed, 2)
    }

    results.append(result)

df = pd.DataFrame(results)
print(df.to_string(index=False))

print("\n===== EVALUATION SUMMARY =====")
print(f"Model: {MODEL}")
print(f"Mean ROUGE-1: {df['ROUGE-1'].mean() * 100:.2f}%")
print(f"Mean ROUGE-2: {df['ROUGE-2'].mean() * 100:.2f}%")
print(f"Mean ROUGE-L: {df['ROUGE-L'].mean() * 100:.2f}%")
print(f"Average response time: {df['Time (seconds)'].mean():.2f}s")


 Test                                                                                                                                                                                   Generated Summary  ROUGE-1  ROUGE-2  ROUGE-L  Time (seconds)
    1                 Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data and make predictions without being explicitly programmed for every task. 0.682927 0.461538 0.634146           75.41
    2 Python is a programming language widely used in data science, utilizing tools such as Pandas to manipulate datasets, NumPy to support numerical computing, and Matplotlib to create visualizations. 0.558140 0.195122 0.511628           75.31
    3                     A train-test split separates data into training and testing sets; the model learns from the training data, and the test data estimates how well it performs on unseen examples. 0.553191 0.222222 0.382979           75.26

===== EVALUATION SU

In [5]:
import ollama
import pandas as pd
import time
from rouge_score import rouge_scorer

MODEL = "gemma4"

# Small sample evaluation dataset
test_data = [
    {
        "text": "Machine learning enables computers to learn patterns "
                "from data. It is used in healthcare, finance, and education.",
        "reference": "Machine learning learns patterns from data and is used "
                    "in healthcare, finance, and education."
    },
    {
        "text": "Python is a popular programming language. It supports data "
                "analysis, machine learning, automation, and web development.",
        "reference": "Python is used for data analysis, machine learning, "
                    "automation, and web development."
    }
]

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

# Candidate hyperparameter combinations
configs = [
    {"temperature": 0.0, "top_k": 20, "top_p": 0.8},
    {"temperature": 0.2, "top_k": 30, "top_p": 0.9},
    {"temperature": 0.5, "top_k": 40, "top_p": 0.9},
    {"temperature": 0.7, "top_k": 50, "top_p": 0.95}
]

results = []

for config in configs:
    rouge_scores = []
    times = []

    for item in test_data:
        start = time.time()

        response = ollama.chat(
            model=MODEL,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Summarize faithfully. Do not add facts. "
                        "Keep the summary concise."
                    )
                },
                {
                    "role": "user",
                    "content": "Summarize this text:\n" + item["text"]
                }
            ],
            options={
                **config,
                "num_predict": 150
            }
        )

        elapsed = time.time() - start
        summary = response["message"]["content"]

        scores = scorer.score(item["reference"], summary)
        rouge_scores.append({
            key: scores[key].fmeasure
            for key in ["rouge1", "rouge2", "rougeL"]
        })
        times.append(elapsed)

    results.append({
        **config,
        "ROUGE-1": 100 * sum(s["rouge1"] for s in rouge_scores)
                       / len(rouge_scores),
        "ROUGE-2": 100 * sum(s["rouge2"] for s in rouge_scores)
                       / len(rouge_scores),
        "ROUGE-L": 100 * sum(s["rougeL"] for s in rouge_scores)
                       / len(rouge_scores),
        "Avg_Time_Seconds": sum(times) / len(times)
    })

df_results = pd.DataFrame(results)
print(df_results.round(2))

# Select the configuration with the highest average ROUGE-L
best_config = max(results, key=lambda x: x["ROUGE-L"])

print("\nBest configuration by ROUGE-L:")
print(best_config)

   temperature  top_k  top_p  ROUGE-1  ROUGE-2  ROUGE-L  Avg_Time_Seconds
0          0.0     20   0.80     0.00     0.00     0.00             27.09
1          0.2     30   0.90    42.86    38.46    42.86             15.98
2          0.5     40   0.90    41.94    34.48    41.94             18.67
3          0.7     50   0.95    42.86    38.46    42.86             20.94

Best configuration by ROUGE-L:
{'temperature': 0.2, 'top_k': 30, 'top_p': 0.9, 'ROUGE-1': 42.857142857142854, 'ROUGE-2': 38.46153846153846, 'ROUGE-L': 42.857142857142854, 'Avg_Time_Seconds': 15.984776616096497}


In [6]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Example dataset
data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
predictions = best_model.predict(X_test)

print("Best parameters:", grid.best_params_)
print("Best CV accuracy:", round(grid.best_score_ * 100, 2), "%")
print("Test accuracy:", round(accuracy_score(y_test, predictions) * 100, 2), "%")
print(classification_report(y_test, predictions, target_names=data.target_names))

Best parameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best CV accuracy: 96.04 %
Test accuracy: 95.61 %
              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

